# CNMC Dataset Preparation for FLEX-Med (NON-IID with Dirichlet Partitioning)\n\nThis notebook prepares the **complete CNMC dataset** for federated learning with **non-IID data distribution**.\n\n⚠️ **KEY DIFFERENCE**: Uses Dirichlet partitioning to simulate heterogeneous hospital populations.\n\n**Output Location:**\n- **NEW**: `/FLEX-Med/cnmc_datasets_noniid/` (created by this notebook)\n- **UNTOUCHED**: `/FLEX-Med/datasets/` and `/FLEX-Med/cnmc_datasets/` (existing datasets remain unchanged)\n\n**Dirichlet Partitioning Strategy:**\n- Controlled by alpha parameter: 0.1 (extreme), 0.5 (moderate ✓), 1.0 (mild), 10.0 (nearly IID)\n- Example: Hospital A might have 75% leukemia, Hospital B might have 75% healthy\n\n**Why Non-IID Matters:**\n- Proves FedMD handles real-world heterogeneity\n- More realistic than balanced (IID) splits\n- Demonstrates system robustness

In [ ]:
# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import required libraries
import os
import shutil
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from collections import Counter
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Libraries imported successfully")

## Configuration

In [ ]:
# Base paths
BASE_PATH = "/content/drive/MyDrive/College"
DATASET_PATH = os.path.join(BASE_PATH, "Datasets/CNMC")  # Fixed: Added CNMC subdirectory
OUTPUT_PATH = os.path.join(BASE_PATH, "FLEX-Med/cnmc_datasets_noniid")  # NEW: Non-IID

# Input paths
TRAINING_DATA_PATH = os.path.join(DATASET_PATH, "training_data")  # Contains fold_0, fold_1, fold_2
VALIDATION_DATA_PATH = os.path.join(DATASET_PATH, "validation_data/C-NMC_test_prelim_phase_data")
VALIDATION_LABELS_CSV = os.path.join(DATASET_PATH, "validation_data/C-NMC_test_prelim_phase_data_labels.csv")
TESTING_DATA_PATH = os.path.join(DATASET_PATH, "testing_data/C-NMC_test_final_phase_data")  # Unlabeled - not used

# Output paths (completely separate from existing datasets/)
PUBLIC_ANCHOR_PATH = os.path.join(OUTPUT_PATH, "cnmc_public_anchor")
PUBLIC_TEST_PATH = os.path.join(OUTPUT_PATH, "cnmc_public_test")

# ============================================================================
# OPTION A: CONSERVATIVE SPLIT (Recommended for FL with proper evaluation)
# ============================================================================
# Split ratios (must sum to 1.0)
PUBLIC_ANCHOR_RATIO = 0.20  # 20% for consensus distillation (shared across clients)
PUBLIC_TEST_RATIO = 0.10    # 10% for unbiased evaluation (never seen during training)
PRIVATE_DATA_RATIO = 0.70   # 70% for client private training data

# Number of clients to create
NUM_CLIENTS = 2

# ============================================================================
# CLIENT DISTRIBUTION: BALANCED (Equal split for now, imbalanced for later)
# ============================================================================
# Options: 'balanced' (equal split), 'imbalanced' (varying sizes), 'custom' (manual ratios)
CLIENT_DISTRIBUTION = 'balanced'  # Currently using balanced for stable training

# For 'custom' distribution, specify ratios for each client (must sum to 1.0)
# Example: [0.6, 0.4] means client 0 gets 60%, client 1 gets 40%
CUSTOM_CLIENT_RATIOS = [0.5, 0.5]

# ============================================================================
# IMPORTANT NOTES:
# - OUTPUT: /FLEX-Med/cnmc_datasets/ (SEPARATE from existing /datasets/)
# - Training data has 3 folds (fold_0, fold_1, fold_2) - will load ALL folds
# - NO ImageNet weights (causes 98.5% healthy, 47% leukemia class imbalance)
# - Random initialization works better for microscopy blood cell images
# - Balanced distribution is ideal for initial FL testing
# - Class imbalance: ~8492 Leukemia, ~4037 Healthy (2.1:1) - handled by Focal Loss
# ============================================================================

print(f"Configuration (Option A - Conservative Split):")
print(f"{'='*60}")
print(f"  Base Path: {BASE_PATH}")
print(f"  Dataset Path: {DATASET_PATH}")
print(f"  Output Path: {OUTPUT_PATH} (SEPARATE DIR)")
print(f"\nInput Paths:")
print(f"  Training: {TRAINING_DATA_PATH} (fold_0, fold_1, fold_2)")
print(f"  Validation: {VALIDATION_DATA_PATH}")
print(f"  Validation CSV: {VALIDATION_LABELS_CSV}")
print(f"\nData Split:")
print(f"  Public Anchor: {PUBLIC_ANCHOR_RATIO*100:.0f}% (for distillation)")
print(f"  Public Test: {PUBLIC_TEST_RATIO*100:.0f}% (for evaluation)")
print(f"  Private Data: {PRIVATE_DATA_RATIO*100:.0f}% (for client training)")
print(f"\nClient Configuration:")
print(f"  Number of Clients: {NUM_CLIENTS}")
print(f"  Distribution: {CLIENT_DISTRIBUTION} (equal split per client)")
print(f"\n⚠️  Note: Output goes to 'cnmc_datasets' (won't touch existing 'datasets')")
print(f"{'='*60}")

# DIRICHLET PARTITIONING CONFIGURATION
# Alpha: 0.1 (extreme), 0.5 (moderate), 1.0 (mild), 10.0 (IID)
DIRICHLET_ALPHA = 0.5

## Step 1: Load Training Data (Labeled via Folder Structure)

In [ ]:
def load_training_data(training_path):
    """
    Load training data from fold structure (fold_0, fold_1, fold_2).
    Each fold contains: all/ and hem/ subdirectories.
    Returns: list of tuples (image_path, label)
    """
    data = []

    # Class mapping: 0 = ALL (Leukemia), 1 = Healthy
    # Note: ImageFolder loads alphabetically, so 'all' = 0, 'hem' = 1
    class_folders = {
        'all': 0,  # ALL/Leukemia
        'hem': 1   # Healthy
    }

    # Find all fold directories (fold_0, fold_1, fold_2, etc.)
    fold_dirs = []
    if os.path.exists(training_path):
        fold_dirs = sorted([d for d in os.listdir(training_path)
                           if os.path.isdir(os.path.join(training_path, d)) and d.startswith('fold_')])

    if not fold_dirs:
        print(f"⚠️  Warning: No fold directories found in {training_path}")
        return data

    print(f"Found {len(fold_dirs)} folds: {fold_dirs}")
    print("")

    # Load data from each fold
    for fold_name in fold_dirs:
        fold_path = os.path.join(training_path, fold_name)
        print(f"Loading {fold_name}:")

        for class_name, label in class_folders.items():
            class_path = os.path.join(fold_path, class_name)

            if not os.path.exists(class_path):
                print(f"  ⚠️  Warning: {class_path} not found")
                continue

            # Get all image files
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.bmp', '.jpg', '.jpeg', '.png'))]

            for img_file in image_files:
                img_path = os.path.join(class_path, img_file)
                data.append((img_path, label))

            print(f"  {class_name}: {len(image_files)} images (label={label})")

        print("")

    return data

print("="*60)
print("Loading training data from all folds...")
print("="*60)
training_data = load_training_data(TRAINING_DATA_PATH)
print(f"✓ Total training samples (all folds): {len(training_data)}")
print("="*60)

## Step 2: Load Validation Data (Labeled via CSV)

In [ ]:
def load_validation_data(validation_path, csv_path):
    """
    Load validation data using CSV labels.
    CSV format: Patient_ID, new_names, labels
    Returns: list of tuples (image_path, label)
    """
    data = []

    # Read CSV
    if not os.path.exists(csv_path):
        print(f"⚠️  Warning: CSV not found at {csv_path}")
        return data

    df = pd.read_csv(csv_path)
    print(f"\nCSV Preview:")
    print(df.head())
    print(f"\nCSV Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

    # Clean column names (remove extra spaces)
    df.columns = df.columns.str.strip()

    # Map labels to match training data convention
    # CSV: 1 = ALL, 0 = Healthy (needs inversion to match our convention)
    label_counts = Counter()

    for _, row in df.iterrows():
        # Get image filename from CSV
        if 'new_names' in df.columns:
            img_filename = row['new_names']
        elif 'Patient_ID' in df.columns:
            img_filename = row['Patient_ID']
        else:
            print(f"⚠️  Error: Cannot find image filename column in CSV")
            continue

        img_path = os.path.join(validation_path, img_filename)

        # Check if image exists
        if not os.path.exists(img_path):
            # Try with .bmp extension if not found
            if not img_filename.lower().endswith('.bmp'):
                img_path = os.path.join(validation_path, img_filename + '.bmp')

            if not os.path.exists(img_path):
                continue

        # Get label from CSV (1 = ALL, 0 = Healthy)
        label = int(row['labels'])

        # Invert label to match our convention (0 = ALL, 1 = Healthy)
        label = 1 - label  # CSV: 1 -> 0 (ALL), CSV: 0 -> 1 (Healthy)

        data.append((img_path, label))
        label_counts[label] += 1

    print(f"\nLoaded validation data:")
    print(f"  ALL (Leukemia, label=0): {label_counts[0]} images")
    print(f"  Healthy (label=1): {label_counts[1]} images")

    return data

print("\n" + "="*60)
print("Loading validation data from CSV...")
print("="*60)
validation_data = load_validation_data(VALIDATION_DATA_PATH, VALIDATION_LABELS_CSV)
print(f"\n✓ Total validation samples: {len(validation_data)}")
print("="*60)

## Step 3: Combine All Labeled Data

In [ ]:
# Combine training and validation data
all_data = training_data + validation_data

# Convert to numpy arrays for easier manipulation
image_paths = np.array([item[0] for item in all_data])
labels = np.array([item[1] for item in all_data])

# Count class distribution
label_counts = Counter(labels)

print(f"\n{'='*60}")
print(f"Combined Dataset Statistics:")
print(f"{'='*60}")
print(f"Training data (all folds): {len(training_data)} samples")
print(f"Validation data:           {len(validation_data)} samples")
print(f"{'='*60}")
print(f"Total combined samples:    {len(all_data)}")
print(f"\nClass Distribution:")
print(f"  ALL (Leukemia, label=0): {label_counts[0]} ({label_counts[0]/len(all_data)*100:.1f}%)")
print(f"  Healthy (label=1):       {label_counts[1]} ({label_counts[1]/len(all_data)*100:.1f}%)")
print(f"\nClass balance ratio: {min(label_counts.values())/max(label_counts.values()):.2f}")
print(f"  (1.0 = perfect balance, >0.8 = good balance)")
print(f"{'='*60}")

## Step 4: Visualize Dataset Distribution

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
class_names = ['ALL\n(Leukemia)', 'Healthy']
counts = [label_counts[0], label_counts[1]]
colors = ['#e74c3c', '#2ecc71']

axes[0].bar(class_names, counts, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_title('Combined CNMC Dataset Distribution', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

for i, (name, count) in enumerate(zip(class_names, counts)):
    axes[0].text(i, count + 50, str(count), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=class_names, autopct='%1.1f%%', colors=colors,
            startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✓ Dataset is {'balanced' if abs(counts[0] - counts[1]) / max(counts) < 0.1 else 'imbalanced'}")

## Step 5: Create Stratified Splits

## Step 5: Dirichlet Partitioning for Client Data (NON-IID)\n\nUses Dirichlet distribution to create heterogeneous data splits.\n\n**How it works:**\n1. For each class, sample proportions from Dirichlet(alpha)\n2. Lower alpha = more skewed = more heterogeneous\n3. Result: Different clients have different class distributions

In [ ]:
def dirichlet_partition(y_private, num_clients, alpha):
    """Partition data using Dirichlet distribution for non-IID splits."""
    np.random.seed(42)
    classes = np.unique(y_private)
    client_indices = [[] for _ in range(num_clients)]
    
    for class_label in classes:
        class_indices = np.where(y_private == class_label)[0]
        np.random.shuffle(class_indices)
        proportions = np.random.dirichlet([alpha] * num_clients)
        counts = (proportions * len(class_indices)).astype(int)
        counts[-1] = len(class_indices) - counts[:-1].sum()
        
        start_idx = 0
        for client_id in range(num_clients):
            end_idx = start_idx + counts[client_id]
            client_indices[client_id].extend(class_indices[start_idx:end_idx])
            start_idx = end_idx
        
        class_name = "ALL" if class_label == 0 else "Healthy"
        print(f"\nClass '{class_name}' distribution (alpha={alpha}):")
        for client_id in range(num_clients):
            pct = (counts[client_id] / len(class_indices)) * 100
            print(f"  Client {client_id}: {counts[client_id]} samples ({pct:.1f}%)")
    
    for client_id in range(num_clients):
        client_indices[client_id] = np.array(client_indices[client_id])
        np.random.shuffle(client_indices[client_id])
    
    return client_indices

print(f"{'='*70}")
print(f"Applying Dirichlet Partitioning (alpha={DIRICHLET_ALPHA})")
print(f"{'='*70}")

# ============================================================================
# MODIFIED: Create Stratified Splits for X_private and y_private
# ============================================================================
print("\nCreating splits for Public Anchor, Public Test, and Private Data...")
# Separate indices by class
indices_all = np.where(labels == 0)[0]
indices_hem = np.where(labels == 1)[0]
# Shuffle indices (random_state=42 for reproducibility)
np.random.seed(42)
np.random.shuffle(indices_all)
np.random.shuffle(indices_hem)
# Calculate subset sizes based on defined ratios
total_samples = len(labels)
n_public_test_target = int(total_samples * PUBLIC_TEST_RATIO)
n_public_anchor_target = int(total_samples * PUBLIC_ANCHOR_RATIO)
# Ensure strictly balanced splits (50% ALL, 50% Healthy) for public sets
# We take N/2 from each class
n_test_per_class = n_public_test_target // 2
n_anchor_per_class = n_public_anchor_target // 2
# Select indices for Public Test (balanced)
test_indices = np.concatenate([
    indices_all[:n_test_per_class],
    indices_hem[:n_test_per_class]
])
X_public_test = image_paths[test_indices]
y_public_test = labels[test_indices]
# Select indices for Public Anchor (balanced)
anchor_indices = np.concatenate([
    indices_all[n_test_per_class:n_test_per_class + n_anchor_per_class],
    indices_hem[n_test_per_class:n_test_per_class + n_anchor_per_class]
])
X_public_anchor = image_paths[anchor_indices]
y_public_anchor = labels[anchor_indices]
# Select remaining indices for Private Data (whatever is left)
private_indices = np.concatenate([
    indices_all[n_test_per_class + n_anchor_per_class:],
    indices_hem[n_test_per_class + n_anchor_per_class:]
])
X_private = image_paths[private_indices]
y_private = labels[private_indices]

client_indices_list = dirichlet_partition(y_private, NUM_CLIENTS, DIRICHLET_ALPHA)

client_datasets = []
for indices in client_indices_list:
    X_client = X_private[indices]
    y_client = y_private[indices]
    client_datasets.append((X_client, y_client))

print(f"\n{'='*70}")
print(f"Client Data Distribution (Non-IID):")
print(f"{'='*70}")

for i, (X_client, y_client) in enumerate(client_datasets):
    all_count = sum(y_client == 0)
    hem_count = sum(y_client == 1)
    total = len(X_client)
    all_pct = (all_count / total) * 100 if total > 0 else 0
    hem_pct = (hem_count / total) * 100 if total > 0 else 0
    balance = min(all_count, hem_count) / max(all_count, hem_count) if max(all_count, hem_count) > 0 else 0
    
    print(f"\nClient {i}:")
    print(f"  Total: {total}")
    print(f"  ALL: {all_count} ({all_pct:.1f}%), Healthy: {hem_count} ({hem_pct:.1f}%)")
    print(f"  Balance ratio: {balance:.2f}")
    
    if balance > 0.8:
        print(f"  → Nearly balanced (IID-like)")
    elif balance > 0.5:
        print(f"  → Moderate heterogeneity ✓")
    else:
        print(f"  → High heterogeneity (extreme)")

print(f"{'='*70}")

## Step 7: Create Output Directory Structure

In [ ]:
def create_dataset_structure(X, y, output_path, description):
    """
    Create ImageFolder structure: output_path/all and output_path/hem
    """
    print(f"\nCreating {description}...")

    # Create directories
    all_dir = os.path.join(output_path, 'all')  # ALL/Leukemia (label=0)
    hem_dir = os.path.join(output_path, 'hem')  # Healthy (label=1)

    os.makedirs(all_dir, exist_ok=True)
    os.makedirs(hem_dir, exist_ok=True)

    # Copy images to appropriate folders
    all_count = 0
    hem_count = 0

    for img_path, label in zip(X, y):
        # Determine destination folder
        if label == 0:  # ALL
            dest_dir = all_dir
            all_count += 1
        else:  # Healthy
            dest_dir = hem_dir
            hem_count += 1

        # Create unique filename
        img_filename = os.path.basename(img_path)
        dest_path = os.path.join(dest_dir, img_filename)

        # Handle duplicate filenames
        counter = 1
        while os.path.exists(dest_path):
            name, ext = os.path.splitext(img_filename)
            dest_path = os.path.join(dest_dir, f"{name}_{counter}{ext}")
            counter += 1

        # Copy image
        try:
            shutil.copy2(img_path, dest_path)
        except Exception as e:
            print(f"  ⚠️  Error copying {img_path}: {e}")

    print(f"  ✓ Created: {output_path}")
    print(f"    ALL: {all_count} images")
    print(f"    Healthy: {hem_count} images")

    return all_count, hem_count

# Clean output directory if it exists
if os.path.exists(OUTPUT_PATH):
    print(f"\n⚠️  Warning: Output directory exists: {OUTPUT_PATH}")
    print("Cleaning existing datasets...")
    shutil.rmtree(OUTPUT_PATH)

os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"\n{'='*60}")
print(f"Creating Output Datasets:")
print(f"{'='*60}")

In [ ]:
# Create public anchor dataset
anchor_all, anchor_hem = create_dataset_structure(
    X_public_anchor, y_public_anchor,
    PUBLIC_ANCHOR_PATH,
    "Public Anchor Dataset (for consensus distillation)"
)

In [ ]:
# Create public test dataset
test_all, test_hem = create_dataset_structure(
    X_public_test, y_public_test,
    PUBLIC_TEST_PATH,
    "Public Test Dataset (for evaluation)"
)

In [ ]:
# Create client datasets
client_stats = []

for i, (X_client, y_client) in enumerate(client_datasets):
    client_path = os.path.join(OUTPUT_PATH, f"cnmc_client_{i}")
    all_count, hem_count = create_dataset_structure(
        X_client, y_client,
        client_path,
        f"Client {i} Private Dataset"
    )
    client_stats.append({
        'client_id': i,
        'total': len(X_client),
        'all': all_count,
        'healthy': hem_count
    })

## Step 8: Create Summary Report

In [ ]:
# Generate summary report
summary_report = {
    'total_samples': len(all_data),
    'public_anchor': {
        'total': len(X_public_anchor),
        'all': anchor_all,
        'healthy': anchor_hem,
        'percentage': len(X_public_anchor) / len(all_data) * 100
    },
    'public_test': {
        'total': len(X_public_test),
        'all': test_all,
        'healthy': test_hem,
        'percentage': len(X_public_test) / len(all_data) * 100
    },
    'clients': client_stats
}

# Save summary as JSON
import json
summary_path = os.path.join(OUTPUT_PATH, 'dataset_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary_report, f, indent=2)

print(f"\n{'='*60}")
print(f"Final Summary:")
print(f"{'='*60}")
print(json.dumps(summary_report, indent=2))
print(f"\n✓ Summary saved to: {summary_path}")

## Step 9: Visualize Final Distribution

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Overall distribution
datasets = ['Public\nAnchor', 'Public\nTest'] + [f'Client {i}' for i in range(NUM_CLIENTS)]
totals = [len(X_public_anchor), len(X_public_test)] + [stats['total'] for stats in client_stats]

axes[0, 0].bar(datasets, totals, color='skyblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Number of Samples', fontsize=12)
axes[0, 0].set_title('Dataset Size Distribution', fontsize=14, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)
for i, (ds, total) in enumerate(zip(datasets, totals)):
    axes[0, 0].text(i, total + 10, str(total), ha='center', fontsize=10, fontweight='bold')

# Plot 2: Class distribution across datasets
x = np.arange(len(datasets))
width = 0.35

all_counts = [anchor_all, test_all] + [stats['all'] for stats in client_stats]
hem_counts = [anchor_hem, test_hem] + [stats['healthy'] for stats in client_stats]

axes[0, 1].bar(x - width/2, all_counts, width, label='ALL (Leukemia)', color='#e74c3c', alpha=0.7)
axes[0, 1].bar(x + width/2, hem_counts, width, label='Healthy', color='#2ecc71', alpha=0.7)
axes[0, 1].set_ylabel('Number of Samples', fontsize=12)
axes[0, 1].set_title('Class Distribution Across Datasets', fontsize=14, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(datasets, rotation=45, ha='right')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Percentage distribution pie chart
sizes = [len(X_public_anchor), len(X_public_test)] + [stats['total'] for stats in client_stats]
labels_pie = ['Public Anchor\n(Distillation)', 'Public Test\n(Evaluation)'] + [f'Client {i}\n(Training)' for i in range(NUM_CLIENTS)]
colors_pie = ['#3498db', '#9b59b6'] + plt.cm.Set3(np.linspace(0, 1, NUM_CLIENTS)).tolist()

axes[1, 0].pie(sizes, labels=labels_pie, autopct='%1.1f%%', colors=colors_pie,
               startangle=90, textprops={'fontsize': 10})
axes[1, 0].set_title('Dataset Usage Distribution', fontsize=14, fontweight='bold')

# Plot 4: Balance ratio per dataset
balance_ratios = []
for all_c, hem_c in zip(all_counts, hem_counts):
    ratio = min(all_c, hem_c) / max(all_c, hem_c) if max(all_c, hem_c) > 0 else 0
    balance_ratios.append(ratio)

colors_bar = ['green' if r > 0.8 else 'orange' if r > 0.6 else 'red' for r in balance_ratios]
axes[1, 1].bar(datasets, balance_ratios, color=colors_bar, alpha=0.7, edgecolor='black')
axes[1, 1].axhline(y=0.8, color='green', linestyle='--', label='Good Balance (>0.8)')
axes[1, 1].axhline(y=0.6, color='orange', linestyle='--', label='Fair Balance (>0.6)')
axes[1, 1].set_ylabel('Balance Ratio', fontsize=12)
axes[1, 1].set_title('Class Balance Per Dataset', fontsize=14, fontweight='bold')
axes[1, 1].set_ylim(0, 1.1)
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualization saved to: {os.path.join(OUTPUT_PATH, 'dataset_distribution.png')}")

## Step 10: Completion Summary

In [ ]:
print(f"\n{'='*70}")
print(f"✓ CNMC Dataset Preparation Complete!")
print(f"{'='*70}")
print(f"\nOutput directory: {OUTPUT_PATH}")
print(f"  (Separate from existing /FLEX-Med/datasets/)")
print(f"\nCreated datasets:")
print(f"  1. cnmc_public_anchor/    - {len(X_public_anchor)} samples (distillation)")
print(f"  2. cnmc_public_test/      - {len(X_public_test)} samples (evaluation)")
for i in range(NUM_CLIENTS):
    print(f"  {i+3}. cnmc_client_{i}/        - {client_stats[i]['total']} samples (training)")

print(f"\n{'='*70}")
print(f"Next Steps to Use CNMC Datasets in FL:")
print(f"{'='*70}")

print(f"\n1. Update task.py paths (ONLY FOR CNMC RUNS):")
print(f"   PUBLIC_ANCHOR_DATASET_PATH = '{PUBLIC_ANCHOR_PATH}'")
print(f"   PUBLIC_TEST_DATASET_PATH = '{PUBLIC_TEST_PATH}'")

print(f"\n2. Create NEW data.json for CNMC (e.g., cnmc_data.json):")
print(f"   [")
for i in range(NUM_CLIENTS):
    client_path = os.path.join(OUTPUT_PATH, f"cnmc_client_{i}")
    model_path = os.path.join(BASE_PATH, f"FLEX-Med/models/cnmc_client_{i}_model.pt")
    print(f"     {{")
    print(f"       \"id\": {i},")
    print(f"       \"client_name\": \"CNMC-Client-{i}\",")
    print(f"       \"model_type\": \"resnet18\",  # or mobilenet_v2")
    print(f"       \"has_local_data\": true,")
    print(f"       \"dataset_path\": \"{client_path}\",")
    print(f"       \"model_path\": \"{model_path}\",")
    print(f"       \"metrics\": \"{{}}\"")
    print(f"     }}{'' if i == NUM_CLIENTS - 1 else ','}")
print(f"   ]")

print(f"\n3. Run FL with CNMC config:")
print(f"   export FLEX_MED_CONFIG_FILE=/path/to/cnmc_data.json")
print(f"   cd federated_learning")
print(f"   flwr run .")

print(f"\n4. Verify Focal Loss handles class imbalance:")
print(f"   Watch for: 'Leukemia: X%, Healthy: Y%, Gap: Z%'")
print(f"   Target: Both >75%, Gap <20%")

print(f"\n{'='*70}")
print(f"Dataset Summary:")
print(f"{'='*70}")
print(f"Total CNMC samples used: {len(all_data)}")
print(f"  Class distribution: ~2.1:1 (Leukemia:Healthy)")
print(f"  Handled by: Focal Loss (alpha=0.25, gamma=2.0)")
print(f"\nPublic Anchor: {len(X_public_anchor)} ({PUBLIC_ANCHOR_RATIO*100:.0f}%)")
print(f"Public Test: {len(X_public_test)} ({PUBLIC_TEST_RATIO*100:.0f}%)")
print(f"Client Training: {len(X_private)} ({PRIVATE_DATA_RATIO*100:.0f}%)")
print(f"  Per client (balanced): ~{len(X_private)//NUM_CLIENTS} samples")
print(f"\n⚠️  This setup is ISOLATED - existing datasets/ directory untouched")
print(f"{'='*70}")